In [ ]:
from google.colab import drive
from pathlib import Path
import os
import subprocess
import logging

drive.mount('/content/drive')

BASE_DIR = Path('/content/drive/MyDrive/DACNTT_voice/Data')
VIDEO_ROOT = BASE_DIR / 'aic'
AUDIO_ROOT = BASE_DIR / 'Audio'

# Cấu hình âm thanh cho Faster-Whisper
SAMPLE_RATE = 16000   # Hz
CHANNELS = 1      # 1 = mono, 2 = stereo

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

print(f"Thư mục video gốc: {VIDEO_ROOT}")
print(f"Thư mục audio đích: {AUDIO_ROOT}")

Mounted at /content/drive
Thư mục video gốc: /content/drive/MyDrive/DACNTT_voice/Data/aic
Thư mục audio đích: /content/drive/MyDrive/DACNTT_voice/Data/Audio


## CODE BỎ QUA NHỮNG FILE ĐÃ TỒN TẠI

In [ ]:
def extract_audio_recursive(video_root, audio_root):
    video_root = Path(video_root)
    audio_root = Path(audio_root)

    # Các định dạng video
    extensions = {'.mp4', '.mov', '.webm'}

    # Tìm tất cả file video trong tất cả các thư mục con
    video_files = [f for f in video_root.rglob('*') if f.suffix.lower() in extensions]

    if not video_files:
        print("Không tìm thấy file video nào!")
        return

    # Kiểm tra các file đã tổn tại - BỎ QUA
    to_process = []
    skipped_count = 0

    print(f"Đang kiểm tra dữ liệu cũ tại {audio_root}...")
    for video_path in video_files:
        relative_path = video_path.relative_to(video_root)
        output_path = audio_root / relative_path.with_suffix('.wav')

        if output_path.exists():
            skipped_count += 1
        else:
            to_process.append((video_path, output_path))

    print(f"--> Đã tồn tại: {skipped_count} file.")
    print(f"--> Xử lý mới: {len(to_process)} file.")

    if not to_process:
        print("Tất cả các file đều đã được xử lý xong từ trước!")
        return

    # Xử lý các file chưa có
    success_count = 0
    error_count = 0

    for video_path, output_path in sorted(to_process):
        output_path.parent.mkdir(parents=True, exist_ok=True)

        command = [
            "ffmpeg", "-y", "-loglevel", "error",
            "-i", str(video_path),
            "-vn",
            "-acodec", "pcm_s16le",
            "-ar", str(SAMPLE_RATE),
            "-ac", str(CHANNELS),
            str(output_path)
        ]

        try:
            subprocess.run(command, check=True)
            success_count += 1

            if success_count % 10 == 0:
                print(f"Đã xử lý: {success_count} file... [{video_path.name}]")

        except Exception as e:
            print(f"Lỗi tại file {video_path.name}: {e}")
            error_count += 1

    print("-" * 30)
    print(f"HOÀN TẤT!")
    print(f"Bỏ qua: {skipped_count} | Thành công: {success_count} | Lỗi: {error_count}")

# Chạy script
extract_audio_recursive(VIDEO_ROOT, AUDIO_ROOT)

Đang kiểm tra dữ liệu cũ tại /content/drive/MyDrive/DACNTT_voice/Data/Audio...
--> Đã tồn tại: 11623 file.
--> Xử lý mới: 0 file.
Tất cả các file đều đã được xử lý xong từ trước!
